# DS_Day01_26: Manufacturing, why is production falling?
**Industry:** Automotive manufacturing

This notebook holds the full analysis and **every visualisation**. It uses the same code (`src/`) as the Streamlit
dashboard (`app.py`), so all numbers match.

**Contents**
0. Setup & data generation
1. Data cleaning & validation
2. Exploratory data analysis (descriptive statistics)
3. When did the decline start?
4. Machines responsible for the largest losses
5. Downtime patterns
6. Shifts & operators (fair comparison)
7. Maintenance vs downtime
8. OEE components
9. Random Forest model & evaluation
10. Key insights
11. Action plan

## 0 · Setup

In [1]:
import sys, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown

from src import analysis as A
from src import model as M
from src.cleaning import clean, validation_checks

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pio.templates.default = "plotly_white"

# Colour palette (colour-blind-safe order), status colours are reserved for maintenance state
BLUE, ORANGE, AQUA, RED, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#e34948", "#a8a69f"
STATUS = {"Up to Date": "#0ca30c", "Due Soon": "#fab219", "Overdue": "#d03b3b", "In Progress": GRAY}
PERIOD = {"Before decline": GRAY, "After decline": BLUE}
SHIFT_C = {"Morning": BLUE, "Afternoon": ORANGE, "Night": AQUA}
OEE_C = {"Availability": BLUE, "Performance": ORANGE, "Quality": AQUA}
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
DIVERGING = [[0, "#a8281f"], [0.5, "#f0efec"], [1, "#256abf"]]

def style(fig, height=420, title=None):
    fig.update_layout(height=height, title=title, margin=dict(l=10, r=10, t=60, b=10),
                      legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0, title=None))
    fig.update_xaxes(showgrid=False)
    return fig

DATA = Path("data/production_data.csv")
if not DATA.exists():
    subprocess.run([sys.executable, "generate_data.py"], check=True)

### The dataset
The data is **synthetic** (see `generate_data.py`): 8 machines × 3 shifts × 365 days in 2025, with realistic data-quality
problems injected on purpose. Column interpretation:

| Column | Meaning |
|---|---|
| Machine_Output | **good** units produced in the shift |
| Production_Volume | **total** units produced (good + rejects) |
| Downtime | minutes stopped in a 480-minute shift |
| Maintenance_Status | Up to Date / Due Soon / Overdue / In Progress (planned maintenance in that shift) |

In [2]:
raw = pd.read_csv(DATA)
print(raw.shape)
display(raw.head(10))
raw.info()

(8865, 9)


,Date,Machine_ID,Machine_Output,Downtime,Shift,Operator,Temperature,Maintenance_Status,Production_Volume
0,2025-08-06,M04,413.0,3.1,Morning,OP02,71.6,Up to Date,424
1,2025-01-20,M07,444.0,26.4,Afternoon,OP07,60.2,Up to Date,450
2,2025-09-16,M02,398.0,34.6,Afternoon,OP06,64.8,Up to Date,407
3,2025-06-03,M01,449.0,13.6,Afternoon,OP07,72.8,Up to Date,460
4,2025-07-17,M01,445.0,18.5,Afternoon,OP06,70.1,Due Soon,456
5,2025-12-11,M06,206.0,268.3,Afternoon,OP09,74.6,Overdue,231
6,2025-04-23,M06,570.0,11.1,Morning,OP05,68.1,Up to Date,584
7,2025-12-15,M01,466.0,8.2,Night,OP11,63.4,Up to Date,472
8,2025-07-12,M07,438.0,11.8,Afternoon,OP07,68.8,Up to Date,447
9,2025-09-03,M04,399.0,11.2,Afternoon,OP07,68.0,Up to Date,409


<class 'pandas.DataFrame'>
RangeIndex: 8865 entries, 0 to 8864
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                8865 non-null   str    
 1   Machine_ID          8865 non-null   str    
 2   Machine_Output      8821 non-null   float64
 3   Downtime            8795 non-null   float64
 4   Shift               8865 non-null   str    
 5   Operator            8777 non-null   str    
 6   Temperature         8645 non-null   float64
 7   Maintenance_Status  8865 non-null   str    
 8   Production_Volume   8865 non-null   int64  
dtypes: float64(3), int64(1), str(5)
memory usage: 912.9 KB


## 1 · Data cleaning & validation

In [3]:
df, log = clean(raw)
display(log)
display(validation_checks(df))

,Step,Rows affected,Action
0,Raw rows loaded,8865,—
1,Shift labels standardised,177,Trimmed whitespace and fixed case (e.g. 'NIGHT...
2,Machine IDs standardised,131,"Trimmed, upper-cased and removed hyphens (e.g...."
3,Maintenance status standardised,87,Mapped to canonical labels
4,Dates in DD/MM/YYYY format,263,Parsed alternate format and unified to ISO dates
5,Unparseable dates dropped,0,Removed
6,Exact duplicate rows removed,105,Double-uploaded records dropped
7,Duplicate Date+Machine+Shift keys removed,0,Kept first record
8,Temperature sensor faults,26,"Values outside 20–120 °C (e.g. 999, -40, 0) se..."
9,Impossible downtime,17,Negative or > 480 min in a 480-min shift set t...


,Check,Passed
0,No missing values in analytical columns,True
1,Downtime within 0–480 min,True
2,Good output ≤ total volume,True
3,Temperature within plausible range,True
4,Unique Date + Machine + Shift key,True
5,Only valid shift labels,True
6,Only valid maintenance labels,True
7,Coverage: 365 days × 8 machines × 3 shifts,True


In [4]:
miss = raw.isna().sum().rename("Missing").to_frame()
miss["% of rows"] = (100 * miss["Missing"] / len(raw)).round(2)
fig = px.bar(miss.reset_index(), x="index", y="% of rows", text="Missing")
fig.update_traces(marker_color=BLUE, textposition="outside")
style(fig, 360, "Missing values in the raw data (count labelled)").update_xaxes(title=None)

In [5]:
# Feature engineering: OEE components, planned/unplanned downtime, days since maintenance, calendar fields
df = A.add_features(df)
df[["Date", "Machine_ID", "Shift", "Downtime", "Planned_Downtime", "Unplanned_Downtime",
    "Days_Since_Maint", "Availability", "Performance", "Quality", "OEE"]].head()

,Date,Machine_ID,Shift,Downtime,Planned_Downtime,Unplanned_Downtime,Days_Since_Maint,Availability,Performance,Quality,OEE
0,2025-01-01,M01,Morning,21.6,0.0,21.6,NaN,0.955000,0.998729,0.981289,0.935940
1,2025-01-01,M01,Afternoon,9.6,0.0,9.6,NaN,0.980000,0.918620,0.980176,0.882401
2,2025-01-01,M01,Night,21.9,0.0,21.9,NaN,0.954375,0.914196,0.986364,0.860589
3,2025-01-01,M02,Morning,2.5,0.0,2.5,NaN,0.994792,0.946308,0.979118,0.921722
4,2025-01-01,M02,Afternoon,0.7,0.0,0.7,NaN,0.998542,0.997439,0.975877,0.971958


## 2 · Exploratory data analysis

In [6]:
num = ["Machine_Output", "Production_Volume", "Downtime", "Temperature", "Availability", "Performance", "Quality", "OEE"]
desc = df[num].describe().T
desc["skew"] = df[num].skew()
desc.round(3)

,count,mean,std,min,25%,50%,75%,max,skew
Machine_Output,8760.0,451.572,83.150,29.000,395.000,435.000,500.000,671.000,0.157
Production_Volume,8760.0,463.541,85.464,32.000,404.000,445.000,518.000,677.000,0.184
Downtime,8760.0,24.046,42.645,0.000,7.300,12.700,21.400,450.000,4.860
Temperature,8760.0,66.990,4.819,53.200,63.500,66.500,70.000,83.400,0.493
Availability,8760.0,0.950,0.089,0.062,0.955,0.974,0.985,1.000,-4.860
Performance,8760.0,0.934,0.036,0.631,0.917,0.939,0.958,1.000,-1.098
Quality,8760.0,0.974,0.020,0.872,0.974,0.979,0.983,0.998,-2.983
OEE,8760.0,0.866,0.099,0.045,0.860,0.890,0.914,0.986,-3.607


In [7]:
per = (df.groupby("Machine_ID")
         .agg(Mean_Output=("Machine_Output", "mean"), SD_Output=("Machine_Output", "std"),
              Mean_Downtime=("Downtime", "mean"), Median_Downtime=("Downtime", "median"),
              Mean_Temp=("Temperature", "mean"), Yield=("Quality", "mean"),
              Overdue_pct=("Maintenance_Status", lambda s: 100 * s.eq("Overdue").mean()))
         .round(2))
per

,Mean_Output,SD_Output,Mean_Downtime,Median_Downtime,Mean_Temp,Yield,Overdue_pct
Machine_ID,,,,,,,
M01,448.47,29.61,19.30,12.5,66.89,0.98,0.82
M02,408.74,24.25,18.46,12.6,62.91,0.98,0.00
M03,551.23,98.01,37.75,14.7,71.05,0.96,38.08
M04,389.45,21.71,17.39,11.9,65.12,0.98,0.00
M05,492.32,35.99,20.41,12.4,67.21,0.98,16.99
M06,521.79,110.21,41.67,13.3,70.59,0.96,43.56
M07,428.60,26.04,18.56,12.6,64.94,0.98,0.00
M08,371.98,24.01,18.82,12.8,67.21,0.98,0.00


In [8]:
fig = px.box(df, x="Machine_ID", y="Machine_Output", points=False, color_discrete_sequence=[BLUE])
style(fig, 400, "Good output per shift by machine")

In [9]:
for col in ["Machine_Output", "Downtime", "Temperature"]:
    fig = px.histogram(df, x=col, nbins=60, color_discrete_sequence=[BLUE])
    style(fig, 320, f"Distribution of {col}").show()

In [10]:
cols = ["Machine_Output", "Downtime", "Unplanned_Downtime", "Temperature", "Days_Since_Maint",
        "Availability", "Performance", "Quality"]
corr = df[cols].corr(method="spearman").round(2)
fig = px.imshow(corr, text_auto=True, zmin=-1, zmax=1, aspect="auto",
                color_continuous_scale=[[0, "#0d366b"], [0.5, "#f0efec"], [1, "#a8281f"]])
style(fig, 480, "Spearman correlation matrix")

## 3 · When did the decline start?
Two independent methods are applied to total daily good output:
1. **Piecewise-linear (hinge) regression**: the breakpoint that minimises squared error.
2. **Control chart**: the first date the 7-day mean falls below baseline − 3σ and stays there for 14 days.

In [11]:
daily = A.daily_plant(df)
decline = A.detect_decline(daily)
D0 = decline["hinge_date"]
df["Period"] = A.split_periods(df, D0)
print(f"Breakpoint (hinge regression): {D0:%d %b %Y}")
print(f"Control-chart signal:          {decline['control_date']:%d %b %Y}")
print(f"Trend before: {decline['slope_before']:+.2f} units/day²,  after: {decline['slope_after']:+.2f} units/day²")

Breakpoint (hinge regression): 28 Jul 2025
Control-chart signal:          24 Aug 2025
Trend before: +0.22 units/day²,  after: -9.94 units/day²


In [12]:
fig = go.Figure()
fig.add_scatter(x=daily["Date"], y=daily["Output"], mode="lines", name="Daily output", line=dict(color=GRAY, width=1))
fig.add_scatter(x=daily["Date"], y=daily["Output_7d"], mode="lines", name="7-day average", line=dict(color=BLUE, width=2))
fig.add_scatter(x=daily["Date"], y=decline["fit"], mode="lines", name="Piecewise-linear fit",
                line=dict(color=ORANGE, width=2, dash="dash"))
fig.add_hline(y=decline["lcl"], line=dict(color=RED, width=1, dash="dot"),
              annotation_text="Lower control limit", annotation_position="bottom left")
fig.add_vline(x=D0, line=dict(color="#52514e", width=1))
fig.add_annotation(x=D0, y=1, yref="paper", text=f"Breakpoint {D0:%d %b}", showarrow=False, xanchor="left", yanchor="top")
fig.update_yaxes(title="Good units / day")
style(fig, 450, "Visualisation ①: Daily plant output, with the decline starting in late July")

In [13]:
losses = A.machine_losses(df, D0)
TOP = losses.head(3)["Machine_ID"].tolist()
mcol = lambda m: {TOP[0]: BLUE, TOP[1]: ORANGE, TOP[2]: AQUA}.get(m, GRAY)

w = df.groupby(["Week", "Machine_ID"])["Machine_Output"].mean().reset_index()
base = df[df["Date"] < "2025-06-01"].groupby("Machine_ID")["Machine_Output"].mean()
w["Index"] = 100 * w["Machine_Output"] / w["Machine_ID"].map(base)
fig = go.Figure()
for m in sorted(w["Machine_ID"].unique(), key=lambda m: m in TOP):
    s = w[w["Machine_ID"] == m]
    fig.add_scatter(x=s["Week"], y=s["Index"], name=m, mode="lines",
                    line=dict(color=mcol(m), width=2.5 if m in TOP else 1), opacity=1 if m in TOP else 0.6)
fig.add_hline(y=100, line=dict(color="#52514e", width=1, dash="dot"))
fig.update_yaxes(title="Index (Jan–May baseline = 100)")
style(fig, 420, "Weekly output by machine, indexed to its own baseline")

In [14]:
mo = daily.groupby(daily["Date"].dt.to_period("M").dt.to_timestamp())["Output"].mean().reset_index()
fig = px.bar(mo, x="Date", y="Output", text="Output")
fig.update_traces(marker_color=[BLUE if d >= pd.Timestamp(D0.year, D0.month, 1) else GRAY for d in mo["Date"]],
                  texttemplate="%{text:,.0f}", textposition="outside")
fig.update_yaxes(title="Mean good units / day", range=[mo["Output"].min() * 0.9, mo["Output"].max() * 1.03])
fig.update_xaxes(dtick="M1", tickformat="%b", title=None)
style(fig, 380, "Monthly mean daily output")

## 4 · Machines responsible for the largest production losses
**Loss** = each machine's expected good output after the decline (its pre-decline average for the same shift) − actual output.

In [15]:
losses.round(1)

,Machine_ID,Units_Lost,Expected,Loss_%,Share_of_Loss_%,Cumulative_%
0,M06,63574.8,273109.8,23.3,52.5,52.5
1,M03,52501.5,282213.5,18.6,43.4,95.9
2,M05,2797.1,233086.1,1.2,2.3,98.2
3,M08,823.7,175554.7,0.5,0.7,98.9
4,M02,514.4,192739.4,0.3,0.4,99.3
5,M07,441.0,202059.0,0.2,0.4,99.7
6,M04,345.8,183579.8,0.2,0.3,100.0
7,M01,-2736.1,210050.9,-1.3,0.0,100.0


In [16]:
L = losses[losses["Units_Lost"] > 0]
fig = go.Figure()
fig.add_bar(x=L["Machine_ID"], y=L["Units_Lost"], marker_color=[mcol(m) for m in L["Machine_ID"]],
            text=[f"{v:,.0f}" for v in L["Units_Lost"]], textposition="outside", name="Units lost")
fig.add_scatter(x=L["Machine_ID"], y=L["Cumulative_%"] / 100 * L["Units_Lost"].sum(), mode="lines+markers",
                name="Cumulative share", line=dict(color="#52514e", width=1.5), marker=dict(size=8),
                text=[f"{c:.0f}%" for c in L["Cumulative_%"]], hovertemplate="%{text}<extra></extra>")
fig.update_yaxes(title="Good units lost since decline")
style(fig, 420, "Visualisation ②: Pareto of lost good units by machine")

In [17]:
comp = A.loss_by_component(df, D0)
c = comp[comp["Machine_ID"].isin(L["Machine_ID"])]
fig = px.bar(c, y="Machine_ID", x="Units_Lost", color="Component", orientation="h", color_discrete_map=OEE_C,
             category_orders={"Machine_ID": losses["Machine_ID"].tolist(), "Component": list(OEE_C)})
fig.update_xaxes(title="Good units lost")
style(fig, 360, "Loss split into OEE components (log decomposition)")

In [18]:
tot = comp.groupby("Component")["Units_Lost"].sum()
(100 * tot / tot.sum()).round(1).rename("% of total loss").to_frame()

,% of total loss
Component,
Availability,46.3
Performance,31.1
Quality,22.6


In [19]:
mm = df.pivot_table(index="Machine_ID", columns="Month", values="Machine_Output", aggfunc="mean")
mm = mm.div(mm.iloc[:, :5].mean(axis=1), axis=0) * 100
mm.columns = [c.strftime("%b") for c in mm.columns]
fig = px.imshow(mm.round(1), text_auto=True, aspect="auto", zmin=70, zmax=110,
                color_continuous_scale=[[0, "#a8281f"], [0.75, "#f0efec"], [1, "#256abf"]])
style(fig, 380, "Machine × month output index (100 = own Jan–May average)")

## 5 · Downtime patterns

In [20]:
display(df.groupby("Period")[["Planned_Downtime", "Unplanned_Downtime", "Breakdown"]].mean().round(2))
h = A.downtime_heatmap(df)
h.columns = [c.strftime("%b") for c in h.columns]
fig = px.imshow(h.round(0), text_auto=True, aspect="auto", color_continuous_scale=SEQ_BLUE)
style(fig, 380, "Mean unplanned downtime (min / shift): machine × month")

,Planned_Downtime,Unplanned_Downtime,Breakdown
Period,,,
After decline,1.07,29.98,0.10
Before decline,1.62,17.15,0.03


In [21]:
wk = df.groupby("Week")[["Planned_Downtime", "Unplanned_Downtime"]].sum().div(60).reset_index()
wk = wk.melt(id_vars="Week", var_name="Type", value_name="Hours")
wk["Type"] = wk["Type"].str.replace("_Downtime", "")
fig = px.bar(wk, x="Week", y="Hours", color="Type", color_discrete_map={"Planned": GRAY, "Unplanned": RED})
fig.update_layout(bargap=0.1)
style(fig, 380, "Weekly plant downtime hours: planned maintenance falls, unplanned stops rise")

In [22]:
br = df.groupby(["Machine_ID", "Period"])["Breakdown"].mean().mul(100).reset_index()
fig = px.bar(br, x="Machine_ID", y="Breakdown", color="Period", barmode="group", color_discrete_map=PERIOD,
             category_orders={"Period": ["Before decline", "After decline"]})
fig.update_yaxes(title=f"% shifts with > {A.UNPLANNED_BREAKDOWN_MIN} min unplanned stop")
style(fig, 380, "Breakdown rate by machine, before vs after")

In [23]:
s = A.downtime_summary(df, ["Shift", "Period"])
fig = px.bar(s, x="Shift", y="Unplanned", color="Period", barmode="group", text="Unplanned", color_discrete_map=PERIOD,
             category_orders={"Period": ["Before decline", "After decline"]})
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
style(fig, 360, "Unplanned downtime by shift").show()

s = A.downtime_summary(df, "DayOfWeek")
fig = px.bar(s, x="DayOfWeek", y="Unplanned", text="Unplanned")
fig.update_traces(marker_color=BLUE, texttemplate="%{text:.1f}", textposition="outside")
style(fig, 360, "Unplanned downtime by day of week")

In [24]:
fig = px.histogram(df, x="Unplanned_Downtime", color="Period", nbins=80, barmode="overlay", opacity=0.65,
                   color_discrete_map=PERIOD, log_y=True)
fig.update_yaxes(title="Shifts (log scale)")
style(fig, 380, "Distribution of unplanned downtime")

Downtime is **similar across shifts and weekdays**. It is concentrated on specific machines and depends on time since the last service.

## 6 · Shifts & operators: a fair comparison
Operators are rostered to fixed shift and machine groups, so **raw averages mostly reflect which machine someone runs**.
Each shift's output is therefore compared with the mean for the *same machine* in the *same week* (and, for operators,
the *same shift*). Operators are shown only by anonymised code, and results describe the **process, not individuals**.

In [25]:
shift_tbl, shift_test = A.group_compare(df, "Shift")
display(shift_tbl.round(3))
print(f"Kruskal-Wallis: H={shift_test['H']:.1f}, p={shift_test['p']:.1e}, effect size eps²={shift_test['eps2']:.3f}")

t = shift_tbl.melt(id_vars="Shift", value_vars=["Raw_vs_Avg_%", "Adj_Mean"], var_name="Measure", value_name="%")
t["Measure"] = t["Measure"].map({"Raw_vs_Avg_%": "Raw vs plant average", "Adj_Mean": "Adjusted (same machine & week)"})
fig = px.bar(t, x="Shift", y="%", color="Measure", barmode="group", text="%", color_discrete_sequence=[GRAY, BLUE])
fig.update_traces(texttemplate="%{text:+.1f}%", textposition="outside")
style(fig, 380, "Shifts: raw vs adjusted output difference")

,Shift,n,Raw_Output,Adj_Mean,Adj_SD,Raw_vs_Avg_%,CI95,Significant
0,Morning,2920,453.954,0.451,10.575,0.528,0.384,True
1,Afternoon,2920,454.340,0.647,8.914,0.613,0.323,True
2,Night,2920,446.422,-1.098,9.002,-1.141,0.327,True


Kruskal-Wallis: H=382.3, p=9.8e-84, effect size eps²=0.043


In [26]:
adj = df.assign(Adj=A.adjusted_residuals(df))
g = adj.groupby(["Period", "Shift"], observed=True)["Adj"].agg(["mean", "std", "size"]).reset_index()
g["ci"] = 1.96 * g["std"] / np.sqrt(g["size"])
fig = px.bar(g, x="Shift", y="mean", color="Period", barmode="group", error_y="ci", color_discrete_map=PERIOD,
             category_orders={"Period": ["Before decline", "After decline"]})
fig.update_yaxes(title="Adjusted % difference (±95% CI)")
style(fig, 380, "Night-shift gap is constant, before and after the decline")

In [27]:
op_tbl, op_test = A.group_compare(df, "Operator", ("Machine_ID", "Week", "Shift"))
mix = A.operator_mix(df, TOP)
o = op_tbl[op_tbl["Operator"] != "Unknown"].merge(mix, on="Operator")
print(f"Operators, Kruskal-Wallis effect size eps² = {op_test['eps2']:.3f}")
display(o[["Operator", "Main_Shift", "Shifts_Worked", "Share_on_Neglected_Machines", "Raw_vs_Avg_%", "Adj_Mean", "CI95"]].round(2))

fig = go.Figure()
fig.add_bar(x=o["Operator"], y=o["Raw_vs_Avg_%"], name="Raw vs plant average", marker_color=GRAY)
fig.add_bar(x=o["Operator"], y=o["Adj_Mean"], name="Adjusted (same machine, week & shift)", marker_color=BLUE,
            error_y=dict(type="data", array=o["CI95"], thickness=1, width=3))
fig.update_layout(barmode="group")
fig.update_yaxes(title="% difference in good output")
style(fig, 420, "Operators: raw differences disappear after adjusting for machine assignment")

Operators, Kruskal-Wallis effect size eps² = 0.023


,Operator,Main_Shift,Shifts_Worked,Share_on_Neglected_Machines,Raw_vs_Avg_%,Adj_Mean,CI95
0,OP01,Morning,715,0.00,-7.66,0.04,0.54
1,OP02,Morning,730,0.00,-7.71,-0.04,0.54
2,OP03,Morning,477,75.68,8.78,0.21,1.14
3,OP04,Morning,463,76.46,9.17,-0.12,1.23
4,OP05,Morning,505,72.87,8.46,0.11,1.04
5,OP06,Afternoon,745,0.00,-7.05,-0.39,0.34
6,OP07,Afternoon,701,0.00,-5.76,0.46,0.36
7,OP08,Afternoon,484,75.21,6.30,-0.94,0.97
8,OP09,Afternoon,475,74.11,8.34,0.86,1.00
9,OP10,Afternoon,479,75.37,8.33,0.17,0.97


## 7 · Maintenance vs downtime

In [28]:
maint = A.maintenance_vs_downtime(df)
display(maint["by_age"].round(3))
print(f"Spearman rho (age vs unplanned downtime) = {maint['rho']:.3f}, p = {maint['p']:.1e}")
print(f"Mann-Whitney, Overdue > Up to Date: p = {maint['mw_p']:.1e}")

,Age_Bin,Unplanned,Breakdown_Rate,Temperature,Quality,n
0,0–14,17.174,0.030,65.849,0.980,3598
1,15–29,16.788,0.027,66.197,0.980,3360
2,30–44,17.567,0.033,68.241,0.978,423
3,45–59,26.844,0.108,71.075,0.966,195
4,60–89,35.986,0.167,74.717,0.949,180
5,90–119,66.012,0.294,76.400,0.925,180
6,120–149,92.109,0.450,77.628,0.903,180
7,150–199,118.358,0.538,77.931,0.893,210


Spearman rho (age vs unplanned downtime) = 0.093, p = 2.1e-17
Mann-Whitney, Overdue > Up to Date: p = 1.3e-52


In [29]:
b = maint["by_age"]
fig = go.Figure()
fig.add_bar(x=b["Age_Bin"].astype(str), y=b["Unplanned"], marker_color=[BLUE if i < 2 else RED for i in range(len(b))],
            text=[f"{v:.0f}" for v in b["Unplanned"]], textposition="outside")
fig.add_vline(x=1.5, line=dict(color="#52514e", width=1, dash="dot"))
fig.add_annotation(x=1.5, y=1, yref="paper", text="30-day interval", showarrow=False, xanchor="left", yanchor="top")
fig.update_xaxes(title="Days since last planned maintenance", type="category")
fig.update_yaxes(title="Mean unplanned downtime (min / shift)")
style(fig, 420, "Visualisation ③: Unplanned downtime by days since last maintenance")

In [30]:
fig = go.Figure()
fig.add_scatter(x=b["Age_Bin"].astype(str), y=100 * b["Breakdown_Rate"], mode="lines+markers",
                line=dict(color=RED, width=2), marker=dict(size=9), name="Breakdown rate")
fig.update_xaxes(title="Days since last planned maintenance", type="category")
fig.update_yaxes(title="% of shifts with a breakdown")
style(fig, 360, "Breakdown probability by maintenance age")

In [31]:
s = maint["by_status"].set_index("Maintenance_Status").loc[["Up to Date", "Due Soon", "Overdue", "In Progress"]].reset_index()
t = s.melt(id_vars="Maintenance_Status", value_vars=["Planned", "Unplanned"], var_name="Type", value_name="min")
fig = px.bar(t, x="Maintenance_Status", y="min", color="Type", color_discrete_map={"Planned": GRAY, "Unplanned": RED})
fig.update_yaxes(title="Mean downtime (min / shift)")
style(fig, 380, "Downtime by recorded maintenance status")

In [32]:
od = maint["overdue_share"]
fig = go.Figure()
for m in sorted(od["Machine_ID"].unique(), key=lambda m: m in TOP):
    x = od[od["Machine_ID"] == m]
    fig.add_scatter(x=x["Month"], y=x["Overdue"], name=m, mode="lines+markers",
                    line=dict(color=mcol(m), width=2.5 if m in TOP else 1))
fig.update_yaxes(title="% of shifts overdue")
style(fig, 380, "Share of shifts running with overdue maintenance")

In [33]:
display(maint["season_check"].round(1))
sc = maint["season_check"].reset_index().melt(id_vars="Season", var_name="Status", value_name="min")
fig = px.bar(sc, x="Season", y="min", color="Status", barmode="group", text="min", color_discrete_map=STATUS)
fig.update_traces(texttemplate="%{text:.0f}", textposition="outside")
style(fig, 380, "Seasonal control: overdue machines have more downtime in every season")

Maintenance_Status,Overdue,Up to Date
Season,,
Jun–Sep (hot),28.3,16.8
Other months,83.5,17.2


In [34]:
d = maint["data"]
d = d[d["Machine_ID"].isin(TOP)].sample(3000, random_state=1)
fig = px.scatter(d, x="Days_Since_Maint", y="Temperature", color="Machine_ID", opacity=0.5,
                 color_discrete_map={m: mcol(m) for m in TOP})
style(fig, 400, "Temperature rises with maintenance age (a symptom of wear)")

## 8 · Overall Equipment Effectiveness (OEE)
| Component | Definition |
|---|---|
| Availability | (480 − Downtime) / 480 |
| Performance | Production_Volume / (Ideal rate × Run time), where ideal rate = machine's 99th-percentile units per running minute |
| Quality | Machine_Output / Production_Volume |
| OEE | A × P × Q |

In [35]:
oee_pp = A.oee_table(df, "Period")
display(oee_pp)
t = oee_pp.melt(id_vars="Period", var_name="Metric", value_name="Value")
fig = px.bar(t, x="Metric", y="Value", color="Period", barmode="group", text="Value", color_discrete_map=PERIOD,
             category_orders={"Period": ["Before decline", "After decline"]})
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_yaxes(title="%", range=[70, 102])
style(fig, 420, "Visualisation ④: OEE components before vs after the decline")

,Period,Availability,Performance,Quality,OEE
0,After decline,93.5,92.4,96.6,83.9
1,Before decline,96.1,94.2,98.0,88.7


In [36]:
mo = A.oee_table(df, "Month").melt(id_vars="Month", var_name="Metric", value_name="Value")
fig = px.line(mo, x="Month", y="Value", color="Metric", markers=True, color_discrete_map={**OEE_C, "OEE": "#52514e"})
fig.update_xaxes(dtick="M1", tickformat="%b")
fig.update_yaxes(title="%")
style(fig, 400, "Monthly OEE and its components")

In [37]:
t = A.oee_table(df, ["Machine_ID", "Period"])
wide = t.pivot(index="Machine_ID", columns="Period")
wide.columns = [f"{a} ({'pre' if b == 'Before decline' else 'post'})" for a, b in wide.columns]
for k in ["Availability", "Performance", "Quality", "OEE"]:
    wide[f"Δ {k}"] = wide[f"{k} (post)"] - wide[f"{k} (pre)"]
wide = wide.sort_values("Δ OEE")
display(wide.round(1))
fig = go.Figure(go.Bar(y=wide.index, x=wide["Δ OEE"], orientation="h", text=[f"{v:+.1f}" for v in wide["Δ OEE"]],
                       textposition="outside", marker_color=[RED if v < -2 else GRAY for v in wide["Δ OEE"]]))
fig.update_xaxes(title="Change in OEE (percentage points)")
style(fig, 360, "Change in OEE by machine")

,Availability (post),Availability (pre),Performance (post),Performance (pre),Quality (post),Quality (pre),OEE (post),OEE (pre),Δ Availability,Δ Performance,Δ Quality,Δ OEE
Machine_ID,,,,,,,,,,,,
M06,84.7,96.3,87.0,94.6,92.2,98.0,68.5,89.2,-11.6,-7.6,-5.8,-20.7
M03,87.0,96.0,88.3,94.2,93.3,98.0,72.2,88.6,-9.0,-5.9,-4.7,-16.4
M05,95.6,95.8,92.5,93.1,97.6,98.0,86.4,87.4,-0.2,-0.6,-0.4,-1.0
M08,95.9,96.2,93.1,93.4,98.0,98.0,87.6,88.0,-0.3,-0.3,0.0,-0.4
M02,96.0,96.3,94.7,94.7,98.0,98.0,89.1,89.4,-0.3,0.0,0.0,-0.3
M04,96.2,96.5,94.4,94.3,98.0,98.0,89.0,89.2,-0.3,0.1,0.0,-0.2
M07,96.1,96.2,94.4,94.5,98.0,98.0,88.9,89.1,-0.1,-0.1,0.0,-0.2
M01,96.6,95.5,94.7,94.4,98.0,98.0,89.6,88.4,1.1,0.3,0.0,1.2


## 9 · Random Forest model
**Target:** good output per shift (`Machine_Output`), predicted from conditions known *before* the shift:
Machine_ID, Shift, Operator, Maintenance_Status, Day of week, Temperature, Days since maintenance.
Downtime, Production_Volume and OEE components are **excluded** because they would leak the answer.
**Evaluation:** machine-stratified 80/20 hold-out, 5-fold cross-validation, and comparison with a naive machine × shift average.

In [38]:
res = M.train_and_evaluate(df)
metrics = pd.DataFrame({"Random Forest (test)": res["rf_metrics"], "Naive baseline (test)": res["baseline_metrics"],
                        "Random Forest (train)": res["train_metrics"]}).T.round(3)
display(metrics)
print("5-fold CV R²:", np.round(res["cv_r2"], 3), f"mean = {res['cv_r2'].mean():.3f} ± {res['cv_r2'].std():.3f}")

,MAE,RMSE,R²,MAPE_%
Random Forest (test),22.255,41.452,0.753,7.836
Naive baseline (test),30.758,59.462,0.492,10.946
Random Forest (train),17.251,31.317,0.858,5.139


5-fold CV R²: [0.778 0.763 0.751 0.75  0.798] mean = 0.768 ± 0.018


In [39]:
imp = res["importance"].sort_values("Importance")
fig = go.Figure(go.Bar(y=imp["Feature"], x=imp["Importance"], orientation="h",
                       error_x=dict(type="data", array=imp["Std"], thickness=1),
                       marker_color=[BLUE if f in ("Days_Since_Maint", "Maintenance_Status") else GRAY for f in imp["Feature"]]))
fig.update_xaxes(title="Increase in MAE when shuffled (units)")
style(fig, 380, "Permutation importance (test set)")

In [40]:
t = res["test"]
fig = px.scatter(t, x="Predicted", y="Machine_Output", color="Maintenance_Status", opacity=0.55, color_discrete_map=STATUS)
lim = [t[["Predicted", "Machine_Output"]].min().min(), t[["Predicted", "Machine_Output"]].max().max()]
fig.add_scatter(x=lim, y=lim, mode="lines", line=dict(color="#52514e", dash="dot", width=1), name="Perfect prediction")
fig.update_yaxes(title="Actual good output")
style(fig, 450, "Actual vs predicted good output (test set)")

In [41]:
fig = go.Figure()
for m in TOP:
    row = df[(df["Machine_ID"] == m) & (df["Shift"] == "Morning")].iloc[[0]].copy()
    row["Temperature"] = df.loc[df["Machine_ID"] == m, "Temperature"].median()
    wi = M.what_if(res["model"], row, list(range(0, 181, 10)))
    fig.add_scatter(x=wi["Days_Since_Maint"], y=wi["Predicted_Output"], name=m, mode="lines+markers",
                    line=dict(color=mcol(m), width=2))
fig.add_vline(x=30, line=dict(color="#52514e", dash="dot", width=1))
fig.update_xaxes(title="Days since last maintenance")
fig.update_yaxes(title="Predicted good units / shift (Morning)")
style(fig, 400, "What-if: predicted output as maintenance is delayed")

In [42]:
f = res["forward"]
print("Forward-in-time test (train Jan–Sep, test Oct–Dec)")
display(pd.DataFrame({"Random Forest": f["rf"], "Naive baseline": f["baseline"]}).T.round(3))
print(f"Max maintenance age in training: {f['max_age_train']:.0f} days, in test: {f['max_age_test']:.0f} days")

Forward-in-time test (train Jan–Sep, test Oct–Dec)


,MAE,RMSE,R²,MAPE_%
Random Forest,40.070,83.658,-0.306,19.254
Naive baseline,55.266,107.161,-1.142,24.312


Max maintenance age in training: 102 days, in test: 194 days


**Interpretation.** The model explains about 75% of shift-to-shift variation (baseline: about 49%), and the train/test gap
is small. Machine_ID ranks first because the machines have different rated speeds. Among the factors the plant can change,
**maintenance age** matters most. Operator, shift and day of week add almost nothing.
**Limitation:** Random Forests cannot extrapolate. In Oct–Dec the frozen machines reached maintenance ages never seen in
Jan–Sep, so a forward-in-time model under-predicts the damage. Retrain it regularly and keep machines inside the
service window it has learned from.

## 10 · Key insights

In [43]:
insights = A.build_insights(df, decline, losses, comp, (shift_tbl, shift_test), (op_tbl, op_test), maint, oee_pp)
display(Markdown("\n\n".join(f"**{i}. {x['title']}**  \n{x['text']}" for i, x in enumerate(insights, 1))))

**1. When the decline started**  
Plant output was flat until about **28 Jul 2025**; after that it fell by about **9.9 units/day each day**. The control-chart rule separately flags a sustained drop from **24 Aug 2025**. Average daily output went from **11,162** before the drop to **9,926** in the last 60 days (**-11.1%**).

**2. Three machines account for most of the lost output**  
**M06, M03, M05** account for **98%** of the good units lost after the decline started (**120,998** units in total). M06 alone ran **23.3%** below its own baseline. The other machines stayed close to their baselines.

**3. The loss is mostly downtime**  
Splitting the loss into OEE components: **Availability 46%**, Quality 23%, Performance 31%. Plant OEE fell from **88.7%** to **83.9%**.

**4. Downtime rises steeply once a machine passes its 30-day service interval**  
Machines serviced in the last 0–14 days lose **17 min** a shift to unplanned stops, with a breakdown in 3% of shifts. At 150–199 days since service this becomes **118 min** and **54%**. The difference between overdue and up-to-date machines is statistically significant (Mann-Whitney p = 1.3e-52). Neglected machines also run hotter (65.8 → 77.9 °C) and scrap more parts (yield 98.0% → 89.3%).

**5. Summer heat is not the explanation**  
All machines ran hotter in summer, but only the ones overdue for service lost output. Within the same season, overdue machines lost **28 vs 17 min** a shift in Jun–Sep and **84 vs 17 min** in the other months. Downtime got worse into the cooler Oct–Dec months, which points to maintenance age as the driver, not ambient heat.

**6. Shifts and operators are not the cause**  
Night shift produces **-1.1%** compared with the same machine in the same week before the decline, and **-1.1%** after it. The gap is constant and did not drive the decline. Raw operator averages differ by up to **±9%**, but only because operators are rostered to different machine groups. Compared with the same machine, week and shift, every operator is within **±1.6%**. The data does not support blaming any individual.

## 11 · Action plan
| # | Action | Owner | When | Target KPI |
|---|---|---|---|---|
| 1 | Service the three affected machines immediately (M06 and M03 first, then M05) | Maintenance lead | 2 weeks | Unplanned downtime < 20 min/shift |
| 2 | Restore the 30-day preventive-maintenance interval, and escalate any machine that is overdue | Plant manager | This month | 0 machines > 35 days since service |
| 3 | Business case: the output lost is far larger than the maintenance hours saved | Finance + Ops | This month | Maintenance budget restored |
| 4 | Condition-based alerts: temperature > 5 °C above the machine's baseline, or > 30 days since service | Controls / IT | 1–2 months | Alerts reviewed daily |
| 5 | Use the Random Forest weekly to rank machines by predicted loss and schedule maintenance | Data / Maintenance | 1–2 months | R² ≥ 0.7, retrained monthly |
| 6 | Track OEE (A × P × Q) daily per machine, with availability as the leading signal | Supervisors | Ongoing | OEE back to the pre-decline level |
| 7 | Review night-shift support as a process question, with no individual blame | HR + Production | Quarterly | Gap closed |
| 8 | Fix data quality at source (date format, drop-downs, sensor range checks, de-duplication) | IT | 1 month | < 0.5% records need cleaning |